<a href="https://colab.research.google.com/github/mathusimon/Data-Visualistions/blob/main/URIP_Phase1_Prototype_Cleaned_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Colab does not have these preinstalled — this takes ~1-2 minutes
!pip -q install osmnx geopandas networkx folium matplotlib shapely pyproj


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 6.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium

from shapely.geometry import Point, Polygon, LineString

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [3]:
# ============================================================
# URIP — Nairobi CBD Study Area
# ============================================================

# Approximate CBD centre
latitude = -1.2864
longitude = 36.8172

# Approximate CBD extent
north = -1.260
south = -1.315
east = 36.850
west = 36.790

print("URIP Study Area: Nairobi CBD")
print(f"Latitude:  {south} to {north}")
print(f"Longitude: {west} to {east}")

URIP Study Area: Nairobi CBD
Latitude:  -1.315 to -1.26
Longitude: 36.79 to 36.85


## 2. Pull the real road network

In [4]:
# ============================================================
# URIP — Nairobi CBD Road Network
# ============================================================

# Nairobi CBD bounding box
north = -1.260
south = -1.315
east = 36.850
west = 36.790

print("Downloading Nairobi CBD road network...")

G = ox.graph_from_bbox(
    bbox=(west, south, east, north),
    network_type="drive"
)

print("Road network downloaded successfully!")
print(f"Nodes: {len(G.nodes):,}")
print(f"Edges: {len(G.edges):,}")


Road network downloaded successfully!
Nodes: 3,078
Edges: 6,711


In [5]:
# ============================================================
# BASELINE ROAD SPEEDS AND TRAVEL TIMES
# ============================================================

G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# Cap anomalous OSM speeds above 80 km/h
edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
edges["speed_kph_original"] = edges["speed_kph"]
edges["speed_flag"] = np.where(edges["speed_kph"] > 80, "anomalous", "valid")
edges["speed_kph"] = edges["speed_kph"].clip(upper=80)
edges["travel_time"] = edges["length"] / (edges["speed_kph"] * 1000 / 3600)

for u, v, k, data in G.edges(keys=True, data=True):
    data["speed_kph"] = edges.loc[(u, v, k), "speed_kph"]
    data["travel_time"] = edges.loc[(u, v, k), "travel_time"]

print("Baseline travel times prepared.")


Baseline travel times prepared.


In [6]:
from google.colab import files

uploaded = files.upload()

Saving URIP_Nairobi_Flood_Scenario.tif to URIP_Nairobi_Flood_Scenario.tif


In [7]:
import rasterio

flood_file = "URIP_Nairobi_Flood_Scenario.tif"

flood_raster = rasterio.open(flood_file)

print("Flood raster loaded")
print("CRS:", flood_raster.crs)
print("Resolution:", flood_raster.res)
print("Bounds:", flood_raster.bounds)
print("Width:", flood_raster.width)
print("Height:", flood_raster.height)

Flood raster loaded
CRS: EPSG:32737
Resolution: (30.0, 30.0)
Bounds: BoundingBox(left=254070.0, bottom=9854520.0, right=260790.0, top=9860640.0)
Width: 224
Height: 204


In [8]:
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape

# Read flood scenario
flood_data = flood_raster.read(1)

# Convert flooded pixels into polygons
flood_shapes = shapes(
    flood_data,
    mask=flood_data == 1,
    transform=flood_raster.transform
)

flood_polygons = [
    shape(geom)
    for geom, value in flood_shapes
    if value == 1
]

# Create one record per polygon
flood_gdf = gpd.GeoDataFrame(
    {
        "flood": [1] * len(flood_polygons)
    },
    geometry=flood_polygons,
    crs=flood_raster.crs
)

print("Flood polygons:", len(flood_gdf))
print("CRS:", flood_gdf.crs)

Flood polygons: 221
CRS: PROJCS["WGS 84 / UTM zone 37S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",39],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32737"]]


In [9]:
edges = ox.graph_to_gdfs(
    G,
    nodes=False,
    edges=True
)

edges_projected = edges.to_crs(flood_gdf.crs)

print("Road segments:", len(edges_projected))
print("Road CRS:", edges_projected.crs)

Road segments: 6711
Road CRS: PROJCS["WGS 84 / UTM zone 37S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",39],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32737"]]


In [10]:
flooded_roads = gpd.sjoin(
    edges_projected,
    flood_gdf,
    how="inner",
    predicate="intersects"
)

print("Flood-affected road segments:", len(flooded_roads))

Flood-affected road segments: 1249


In [11]:
# Dissolve all flood polygons into one flood extent
flood_union = flood_gdf.dissolve()

# Get the single flood geometry
flood_geometry = flood_union.geometry.iloc[0]

# Calculate the portion of each road affected by flooding
edges_projected["flooded_length_m"] = (
    edges_projected.geometry
    .intersection(flood_geometry)
    .length
)

edges_projected["road_length_m"] = edges_projected.geometry.length

edges_projected["flood_percentage"] = (
    edges_projected["flooded_length_m"]
    / edges_projected["road_length_m"]
    * 100
)

print(
    edges_projected["flood_percentage"]
    .describe()
)

count    6711.000000
mean       14.501488
std        33.688476
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: flood_percentage, dtype: float64


In [12]:
# Classify roads by percentage of segment affected

edges_projected["flood_class"] = pd.cut(
    edges_projected["flood_percentage"],
    bins=[-0.01, 0, 20, 50, 100],
    labels=[
        "Unaffected",
        "Minor exposure",
        "Moderate disruption",
        "Major disruption"
    ]
)

print(
    edges_projected["flood_class"]
    .value_counts()
    .sort_index()
)

flood_class
Unaffected             5513
Minor exposure           96
Moderate disruption     131
Major disruption        971
Name: count, dtype: int64


In [13]:
flood_summary = (
    edges_projected["flood_class"]
    .value_counts()
    .sort_index()
)

flood_summary_pct = (
    flood_summary / len(edges_projected) * 100
).round(2)

print("Road disruption summary")
print("=======================")

for category in flood_summary.index:
    print(
        f"{category}: "
        f"{flood_summary[category]} roads "
        f"({flood_summary_pct[category]}%)"
    )

Road disruption summary
Unaffected: 5513 roads (82.15%)
Minor exposure: 96 roads (1.43%)
Moderate disruption: 131 roads (1.95%)
Major disruption: 971 roads (14.47%)


In [14]:
# Identify roads classified as major disruption

major_flood_roads = edges_projected[
    edges_projected["flood_percentage"] > 50
].copy()

print("Major disruption roads:", len(major_flood_roads))

Major disruption roads: 971


In [15]:
# Create flood-adjusted network
G_flood_impedance = G.copy()

# Apply flood penalties to every road segment
for u, v, k, data in G_flood_impedance.edges(
    keys=True,
    data=True
):

    # Get the corresponding road segment
    flood_pct = edges_projected.loc[
        (u, v, k),
        "flood_percentage"
    ]

    # Assign flood multiplier
    if flood_pct == 0:
        multiplier = 1.0
    elif flood_pct <= 20:
        multiplier = 1.25
    elif flood_pct <= 50:
        multiplier = 1.75
    else:
        multiplier = 3.0

    # Modify travel time
    data["flood_multiplier"] = multiplier
    data["flood_travel_time"] = (
        data["travel_time"] * multiplier
    )

print("Flood-adjusted network created.")

Flood-adjusted network created.


In [16]:
from collections import Counter

multiplier_counts = Counter()

for u, v, k, data in G_flood_impedance.edges(
    keys=True,
    data=True
):
    multiplier_counts[data["flood_multiplier"]] += 1

print("Flood multiplier distribution")
print("=============================")

for multiplier, count in sorted(multiplier_counts.items()):
    percentage = count / len(G_flood_impedance.edges) * 100
    print(
        f"{multiplier}x: "
        f"{count} roads "
        f"({percentage:.2f}%)"
    )

Flood multiplier distribution
1.0x: 5513 roads (82.15%)
1.25x: 96 roads (1.43%)
1.75x: 131 roads (1.95%)
3.0x: 971 roads (14.47%)


In [17]:
from google.colab import files

uploaded = files.upload()

Saving HOSP.zip to HOSP.zip


In [18]:
import zipfile
import os

with zipfile.ZipFile("HOSP.zip", "r") as zip_ref:
    zip_ref.extractall("/content/hospital_data")

print(os.listdir("/content/hospital_data"))

['HOSP']


In [19]:
import geopandas as gpd

hospitals = gpd.read_file("/content/hospital_data/HOSP/Hospitals.shp")

print("Number of hospitals:", len(hospitals))
print("CRS:", hospitals.crs)
print("Columns:")
print(hospitals.columns.tolist())

display(hospitals.head())

Number of hospitals: 141
CRS: EPSG:4326
Columns:
['OBJECTID', 'osm_id', 'code', 'fclass', 'name', 'geometry']


,OBJECTID,osm_id,code,fclass,name,geometry
0,1,275554473,2110,hospital,M. P. Shah Hospital,POINT (36.81197 -1.26377)
1,2,612007438,2110,hospital,Upendo Vision Clinic,POINT (36.78605 -1.30787)
2,3,612007453,2110,hospital,New Makina Clinic,POINT (36.78633 -1.3072)
3,4,612007456,2110,hospital,St Augustus Medical Clinic,POINT (36.78882 -1.30723)
4,5,612007539,2110,hospital,Wema Clinic,POINT (36.77823 -1.31793)


In [20]:
# Convert hospitals to the same CRS as the OSM network
hospitals = hospitals.to_crs("EPSG:4326")

# Study area used for the OSM road network
north = -1.260
south = -1.315
east = 36.850
west = 36.790

# Select hospitals inside the URIP study area
hospitals_cbd = hospitals[
    (hospitals.geometry.x >= west) &
    (hospitals.geometry.x <= east) &
    (hospitals.geometry.y >= south) &
    (hospitals.geometry.y <= north)
].copy()

print("Hospitals in study area:", len(hospitals_cbd))
display(hospitals_cbd[["name", "geometry"]])

Hospitals in study area: 34


,name,geometry
0,M. P. Shah Hospital,POINT (36.81197 -1.26377)
5,Saola Health Clinic,POINT (36.7991 -1.31172)
10,Frepals Maternity,POINT (36.79238 -1.31202)
11,Community Royal clinic,POINT (36.79648 -1.31384)
19,University Mobile Clinic,POINT (36.81555 -1.27961)
20,Student's Clinic,POINT (36.81072 -1.27827)
26,Huduma Bora Medics Clinic,POINT (36.84421 -1.26388)
30,Jamko health clinic,POINT (36.79169 -1.29348)
39,RTI health centre,POINT (36.84206 -1.31392)
51,The Aga Khan Medical Centre Tmall,POINT (36.81689 -1.31216)


In [21]:
# Find the nearest OSM road node for each hospital
hospitals_cbd["nearest_node"] = hospitals_cbd.geometry.apply(
    lambda point: ox.distance.nearest_nodes(
        G,
        X=point.x,
        Y=point.y
    )
)

print("Hospitals successfully snapped:", hospitals_cbd["nearest_node"].notna().sum())

display(
    hospitals_cbd[["name", "nearest_node", "geometry"]]
)

Hospitals successfully snapped: 34


,name,nearest_node,geometry
0,M. P. Shah Hospital,7730852179,POINT (36.81197 -1.26377)
5,Saola Health Clinic,7723878225,POINT (36.7991 -1.31172)
10,Frepals Maternity,612008259,POINT (36.79238 -1.31202)
11,Community Royal clinic,612008259,POINT (36.79648 -1.31384)
19,University Mobile Clinic,9764595295,POINT (36.81555 -1.27961)
20,Student's Clinic,293108085,POINT (36.81072 -1.27827)
26,Huduma Bora Medics Clinic,1110239841,POINT (36.84421 -1.26388)
30,Jamko health clinic,2471977785,POINT (36.79169 -1.29348)
39,RTI health centre,280773532,POINT (36.84206 -1.31392)
51,The Aga Khan Medical Centre Tmall,6371398615,POINT (36.81689 -1.31216)


In [22]:
import random
from shapely.geometry import Point

# Reproducible random seed
random.seed(42)

# Get all road nodes from the OSM network
nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)

# Select 50 nodes randomly as simulated emergency incidents
incident_nodes = random.sample(list(nodes.index), 50)

# Create incident GeoDataFrame
incidents = nodes.loc[incident_nodes].copy()

incidents["incident_id"] = range(1, len(incidents) + 1)
incidents["incident_type"] = "Simulated Emergency"

incidents = incidents[
    ["incident_id", "incident_type", "geometry"]
].copy()

print("Number of incidents:", len(incidents))

display(incidents.head())

Number of incidents: 50


,incident_id,incident_type,geometry
osmid,,,
7843839186,1,Simulated Emergency,POINT (36.83184 -1.28356)
280991850,2,Simulated Emergency,POINT (36.80681 -1.26758)
30396828,3,Simulated Emergency,POINT (36.79066 -1.29947)
13946542619,4,Simulated Emergency,POINT (36.81552 -1.29366)
1667938234,5,Simulated Emergency,POINT (36.79509 -1.27312)


In [23]:
import numpy as np
import geopandas as gpd

# Get OSM road nodes
nodes = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=False
).copy()

# Project nodes to the flood raster CRS
nodes_projected = nodes.to_crs(flood_raster.crs)

# Random seed for reproducibility
np.random.seed(42)

# Randomly select 50 road nodes
incident_node_ids = np.random.choice(
    nodes_projected.index,
    size=50,
    replace=False
)

# Create incident GeoDataFrame
incidents = nodes_projected.loc[incident_node_ids].copy()

incidents["incident_id"] = range(1, len(incidents) + 1)
incidents["incident_type"] = "Simulated Emergency"

# Keep only useful fields
incidents = incidents[
    ["incident_id", "incident_type", "geometry"]
].copy()

print("Number of incidents:", len(incidents))

Number of incidents: 50


In [24]:
incident_coords = [
    (point.x, point.y)
    for point in incidents.geometry
]

flood_values = []

for value in flood_raster.sample(incident_coords):
    flood_values.append(int(value[0]))

incidents["flood_exposure"] = flood_values

print(
    incidents["flood_exposure"]
    .value_counts()
    .sort_index()
)

flood_exposure
0    44
1     6
Name: count, dtype: int64


In [25]:
# Convert incidents back to WGS84
incidents_wgs84 = incidents.to_crs("EPSG:4326")

# Make sure hospitals are WGS84
hospitals_cbd = hospitals_cbd.to_crs("EPSG:4326")

print("Hospitals:", len(hospitals_cbd))
print("Incidents:", len(incidents_wgs84))
print("Hospital CRS:", hospitals_cbd.crs)
print("Incident CRS:", incidents_wgs84.crs)

Hospitals: 34
Incidents: 50
Hospital CRS: EPSG:4326
Incident CRS: EPSG:4326


In [26]:
# Snap hospitals to nearest road nodes
hospitals_cbd["nearest_node"] = ox.distance.nearest_nodes(
    G,
    X=hospitals_cbd.geometry.x,
    Y=hospitals_cbd.geometry.y
)

# Snap incidents to nearest road nodes
incidents_wgs84["nearest_node"] = ox.distance.nearest_nodes(
    G,
    X=incidents_wgs84.geometry.x,
    Y=incidents_wgs84.geometry.y
)

print("Hospitals snapped:", hospitals_cbd["nearest_node"].notna().sum())
print("Incidents snapped:", incidents_wgs84["nearest_node"].notna().sum())

Hospitals snapped: 34
Incidents snapped: 50


In [27]:
edges = ox.graph_to_gdfs(
    G,
    nodes=False,
    edges=True
).copy()

print("Number of road segments:", len(edges))

print("\nHighway classifications:")
print(edges["highway"].value_counts().head(20))

Number of road segments: 6711

Highway classifications:
highway
residential                     3594
secondary                       1151
unclassified                     771
tertiary                         572
living_street                    229
trunk                            172
secondary_link                    91
primary                           30
primary_link                      30
tertiary_link                     25
trunk_link                        16
motorway_link                     10
motorway                          10
[residential, unclassified]        4
[secondary, primary]               1
[secondary, unclassified]          1
[residential, living_street]       1
[trunk_link, primary]              1
[living_street, residential]       1
[tertiary, unclassified]           1
Name: count, dtype: int64


In [28]:
# ---------------------------------------------------------
# URIP TRAFFIC MODEL — PEAK HOUR SCENARIO
# ---------------------------------------------------------

G_traffic = G.copy()

# Traffic multipliers by OSM road class
traffic_multipliers = {
    "motorway": 1.20,
    "trunk": 1.30,
    "primary": 1.50,
    "secondary": 1.75,
    "tertiary": 2.00,
    "residential": 1.40,
    "unclassified": 1.30,
    "service": 1.20,
    "living_street": 1.25
}

for u, v, k, data in G_traffic.edges(
    keys=True,
    data=True
):

    road_type = data.get("highway")

    # OSM can sometimes store highway as a list
    if isinstance(road_type, list):
        road_type = road_type[0]

    # Default multiplier if road type isn't in our table
    multiplier = traffic_multipliers.get(
        road_type,
        1.50
    )

    data["traffic_multiplier"] = multiplier

    data["traffic_travel_time"] = (
        data["travel_time"] * multiplier
    )

print("Traffic model created successfully.")

Traffic model created successfully.


In [29]:
# ---------------------------------------------------------
# Identify potential traffic hotspots
# ---------------------------------------------------------

nodes_traffic = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=False
).copy()

# Rank nodes by number of connecting streets
hotspots = nodes_traffic[
    nodes_traffic["street_count"] >= 4
].copy()

print("Potential traffic hotspots:", len(hotspots))

display(
    hotspots[
        ["street_count", "geometry"]
    ].sort_values(
        "street_count",
        ascending=False
    ).head(20)
)

Potential traffic hotspots: 311


,street_count,geometry
osmid,,
4264625840,6,POINT (36.80078 -1.26588)
4264626106,5,POINT (36.80099 -1.26546)
12414258036,5,POINT (36.82788 -1.28163)
6164194108,4,POINT (36.8086 -1.29626)
6126721043,4,POINT (36.81313 -1.26222)
5771203409,4,POINT (36.82781 -1.28187)
5580373449,4,POINT (36.83026 -1.30024)
5496667744,4,POINT (36.83173 -1.28226)
5496066102,4,POINT (36.84733 -1.2945)


In [31]:
from shapely.ops import unary_union

# ---------------------------------------------------------
# Create hotspot zones
# ---------------------------------------------------------

hotspots_utm = hotspots.to_crs("EPSG:32737")

hotspot_50m = hotspots_utm.geometry.buffer(50)
hotspot_100m = hotspots_utm.geometry.buffer(100)

hotspot_50m_union = unary_union(hotspot_50m)
hotspot_100m_union = unary_union(hotspot_100m)

print("Hotspot zones created.")

Hotspot zones created.


In [32]:
# Convert road edges to projected CRS
traffic_edges_utm = ox.graph_to_gdfs(
    G_traffic,
    nodes=False,
    edges=True
).to_crs("EPSG:32737")

for idx, row in traffic_edges_utm.iterrows():

    midpoint = row.geometry.interpolate(0.5, normalized=True)

    if hotspot_50m_union.contains(midpoint):
        hotspot_multiplier = 1.50

    elif hotspot_100m_union.contains(midpoint):
        hotspot_multiplier = 1.25

    else:
        hotspot_multiplier = 1.00

    traffic_edges_utm.loc[
        idx,
        "hotspot_multiplier"
    ] = hotspot_multiplier

    traffic_edges_utm.loc[
        idx,
        "final_traffic_time"
    ] = (
        row["traffic_travel_time"]
        * hotspot_multiplier
    )

print("Hotspot traffic impedance applied.")

Hotspot traffic impedance applied.


In [33]:
for (u, v, k), row in traffic_edges_utm.iterrows():

    if G_traffic.has_edge(u, v, k):

        G_traffic[u][v][k]["hotspot_multiplier"] = (
            row["hotspot_multiplier"]
        )

        G_traffic[u][v][k]["final_traffic_time"] = (
            row["final_traffic_time"]
        )

print("Final traffic travel times written to network.")

Final traffic travel times written to network.


In [34]:
# Create combined peak-traffic + flood network
G_combined = G_traffic.copy()

for u, v, k, data in G_combined.edges(keys=True, data=True):

    # Get flood multiplier from the flood impedance network
    flood_multiplier = G_flood_impedance[u][v][k].get(
        "flood_multiplier", 1.0
    )

    # Combined travel time
    data["combined_flood_multiplier"] = flood_multiplier
    data["combined_travel_time"] = (
        data["final_traffic_time"] * flood_multiplier
    )

print("Combined network created.")

Combined network created.


In [35]:
# Inspect combined travel times
combined_edges = ox.graph_to_gdfs(
    G_combined,
    nodes=False,
    edges=True
)

print(combined_edges[
    [
        "highway",
        "length",
        "travel_time",
        "final_traffic_time",
        "combined_travel_time"
    ]
].head(10))

                              highway       length  travel_time  \
u        v          key                                           
30030168 6364196071 0        tertiary   400.498579    28.835898   
         2469825457 0    unclassified   136.365464     9.217615   
         2469825452 0        tertiary   270.463675    20.148407   
         4816654149 0        tertiary    64.295402     5.786586   
30030170 2469825501 0        tertiary   126.789563     9.128849   
         6164233256 0        tertiary    36.846285     2.652933   
30030172 30030180   0        tertiary  1026.043848    73.875157   
         3393511907 0        tertiary   637.820173   114.807631   
         30030176   0        tertiary   112.337191    20.220694   
30030176 293111184  0       secondary   113.843124     8.141197   

                         final_traffic_time  combined_travel_time  
u        v          key                                            
30030168 6364196071 0             57.671795             57.

In [36]:
# Snap hospitals and incidents to the nearest road-network node

hospitals_cbd = hospitals_cbd.copy()
incidents_wgs84 = incidents_wgs84.copy()

# Hospital nearest road node
hospitals_cbd["nearest_node"] = ox.distance.nearest_nodes(
    G,
    X=hospitals_cbd.geometry.x,
    Y=hospitals_cbd.geometry.y
)

# Incident nearest road node
incidents_wgs84["nearest_node"] = ox.distance.nearest_nodes(
    G,
    X=incidents_wgs84.geometry.x,
    Y=incidents_wgs84.geometry.y
)

print("Hospitals snapped:", len(hospitals_cbd))
print("Incidents snapped:", len(incidents_wgs84))

print("\nHospital sample:")
print(
    hospitals_cbd[
        ["name", "nearest_node"]
    ].head()
)

print("\nIncident sample:")
print(
    incidents_wgs84[
        ["incident_id", "incident_type", "nearest_node"]
    ].head()
)

Hospitals snapped: 34
Incidents snapped: 50

Hospital sample:
                        name  nearest_node
0        M. P. Shah Hospital    7730852179
5        Saola Health Clinic    7723878225
10         Frepals Maternity     612008259
11    Community Royal clinic     612008259
19  University Mobile Clinic    9764595295

Incident sample:
            incident_id        incident_type  nearest_node
osmid                                                     
30214997              1  Simulated Emergency      30214997
9683942781            2  Simulated Emergency    9683942781
6363952419            3  Simulated Emergency    6363952419
268642063             4  Simulated Emergency     268642063
8574526971            5  Simulated Emergency    8574526971


In [37]:
# Calculate emergency-response travel times
# for all hospital → incident combinations

results = []

for _, hospital in hospitals_cbd.iterrows():

    hospital_name = hospital["name"]
    hospital_node = hospital["nearest_node"]

    for _, incident in incidents_wgs84.iterrows():

        incident_id = incident["incident_id"]
        incident_node = incident["nearest_node"]

        record = {
            "hospital": hospital_name,
            "hospital_node": hospital_node,
            "incident_id": incident_id,
            "incident_node": incident_node
        }

        # -----------------------------
        # 1. Baseline scenario
        # -----------------------------
        try:
            baseline_time = nx.shortest_path_length(
                G,
                hospital_node,
                incident_node,
                weight="travel_time"
            )
        except nx.NetworkXNoPath:
            baseline_time = np.nan

        # -----------------------------
        # 2. Peak traffic scenario
        # -----------------------------
        try:
            traffic_time = nx.shortest_path_length(
                G_traffic,
                hospital_node,
                incident_node,
                weight="final_traffic_time"
            )
        except nx.NetworkXNoPath:
            traffic_time = np.nan

        # -----------------------------
        # 3. Peak traffic + flood
        # -----------------------------
        try:
            combined_time = nx.shortest_path_length(
                G_combined,
                hospital_node,
                incident_node,
                weight="combined_travel_time"
            )
        except nx.NetworkXNoPath:
            combined_time = np.nan

        record["baseline_time_sec"] = baseline_time
        record["traffic_time_sec"] = traffic_time
        record["flood_traffic_time_sec"] = combined_time

        results.append(record)

# Convert results to DataFrame
routing_results = pd.DataFrame(results)

print("Total route combinations:", len(routing_results))
print("\nSample results:")
print(routing_results.head())

Total route combinations: 1700

Sample results:
              hospital  hospital_node  incident_id  incident_node  \
0  M. P. Shah Hospital     7730852179            1       30214997   
1  M. P. Shah Hospital     7730852179            2     9683942781   
2  M. P. Shah Hospital     7730852179            3     6363952419   
3  M. P. Shah Hospital     7730852179            4      268642063   
4  M. P. Shah Hospital     7730852179            5     8574526971   

   baseline_time_sec  traffic_time_sec  flood_traffic_time_sec  
0         270.668725        417.101511              619.288998  
1         154.210434        263.229964              278.532939  
2         440.402926        693.048259             1048.141017  
3         276.215487        506.251280              710.868917  
4         259.688314        451.332803              451.332803  


In [38]:
# Calculate response-time impacts

routing_results["baseline_min"] = (
    routing_results["baseline_time_sec"] / 60
)

routing_results["traffic_min"] = (
    routing_results["traffic_time_sec"] / 60
)

routing_results["flood_traffic_min"] = (
    routing_results["flood_traffic_time_sec"] / 60
)

# Traffic delay
routing_results["traffic_delay_sec"] = (
    routing_results["traffic_time_sec"]
    - routing_results["baseline_time_sec"]
)

# Additional flood-related delay
routing_results["flood_delay_sec"] = (
    routing_results["flood_traffic_time_sec"]
    - routing_results["traffic_time_sec"]
)

# Total delay relative to baseline
routing_results["total_delay_sec"] = (
    routing_results["flood_traffic_time_sec"]
    - routing_results["baseline_time_sec"]
)

# Percentage increase relative to baseline
routing_results["total_delay_pct"] = (
    routing_results["total_delay_sec"]
    / routing_results["baseline_time_sec"]
) * 100

print(routing_results[
    [
        "hospital",
        "incident_id",
        "baseline_min",
        "traffic_min",
        "flood_traffic_min",
        "traffic_delay_sec",
        "flood_delay_sec",
        "total_delay_pct"
    ]
].head(10))

              hospital  incident_id  baseline_min  traffic_min  \
0  M. P. Shah Hospital            1      4.511145     6.951692   
1  M. P. Shah Hospital            2      2.570174     4.387166   
2  M. P. Shah Hospital            3      7.340049    11.550804   
3  M. P. Shah Hospital            4      4.603591     8.437521   
4  M. P. Shah Hospital            5      4.328139     7.522213   
5  M. P. Shah Hospital            6      5.854715     9.957514   
6  M. P. Shah Hospital            7      3.564306     6.255317   
7  M. P. Shah Hospital            8      6.159220    11.677019   
8  M. P. Shah Hospital            9      4.047190     6.336345   
9  M. P. Shah Hospital           10      2.891502     4.527577   

   flood_traffic_min  traffic_delay_sec  flood_delay_sec  total_delay_pct  
0          10.321483         146.432786       202.187487       128.799614  
1           4.642216         109.019529        15.302976        80.618737  
2          17.469017         252.645333      

In [39]:
# Find fastest hospital for each incident under each scenario

baseline_best = (
    routing_results
    .loc[routing_results.groupby("incident_id")["baseline_time_sec"].idxmin()]
    .copy()
)

traffic_best = (
    routing_results
    .loc[routing_results.groupby("incident_id")["traffic_time_sec"].idxmin()]
    .copy()
)

flood_best = (
    routing_results
    .loc[routing_results.groupby("incident_id")["flood_traffic_time_sec"].idxmin()]
    .copy()
)

print("Baseline fastest responses:")
print(
    baseline_best[
        ["incident_id", "hospital", "baseline_min"]
    ].head(10)
)

print("\nPeak traffic fastest responses:")
print(
    traffic_best[
        ["incident_id", "hospital", "traffic_min"]
    ].head(10)
)

print("\nPeak traffic + flood fastest responses:")
print(
    flood_best[
        ["incident_id", "hospital", "flood_traffic_min"]
    ].head(10)
)

Baseline fastest responses:
      incident_id                                    hospital  baseline_min
1450            1                              mater hospital      1.349275
1601            2                          Vital Rey Hospital      0.856709
452             3           The Aga Khan Medical Centre Tmall      0.097859
1453            4                              mater hospital      0.514240
1104            5                              Hayat Hospital      0.641796
605             6                         The Jordon Hospital      1.243444
256             7                            Student's Clinic      1.195338
857             8                               Care Hospital      1.174393
1458            9                              mater hospital      1.423532
959            10  AIC Kijabe Hospital Nairobi Medical Centre      0.805933

Peak traffic fastest responses:
      incident_id                                    hospital  traffic_min
1450            1           

In [40]:
from google.colab import files

uploaded = files.upload()

Saving URIP_Nairobi_Flood_Susceptibility.tif to URIP_Nairobi_Flood_Susceptibility.tif


In [41]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.warp import transform_bounds


# ============================================================
# LOAD RASTERS
# ============================================================

flood_raster = rasterio.open(
    "/content/URIP_Nairobi_Flood_Scenario.tif"
)

susceptibility_raster = rasterio.open(
    "/content/URIP_Nairobi_Flood_Susceptibility.tif"
)


# ============================================================
# READ DATA
# ============================================================

flood_data = flood_raster.read(1).astype(float)

susceptibility_data = susceptibility_raster.read(1).astype(float)


# ============================================================
# CONVERT RASTER BOUNDS FROM UTM → LAT/LON
# ============================================================

flood_bounds_wgs84 = transform_bounds(
    flood_raster.crs,
    "EPSG:4326",
    *flood_raster.bounds
)

susceptibility_bounds_wgs84 = transform_bounds(
    susceptibility_raster.crs,
    "EPSG:4326",
    *susceptibility_raster.bounds
)


# ============================================================
# FOLIUM BOUNDS
# ============================================================

flood_bounds = [

    [
        flood_bounds_wgs84[1],
        flood_bounds_wgs84[0]
    ],

    [
        flood_bounds_wgs84[3],
        flood_bounds_wgs84[2]
    ]

]


susceptibility_bounds = [

    [
        susceptibility_bounds_wgs84[1],
        susceptibility_bounds_wgs84[0]
    ],

    [
        susceptibility_bounds_wgs84[3],
        susceptibility_bounds_wgs84[2]
    ]

]


print("Flood bounds:")
print(flood_bounds)

print("\nSusceptibility bounds:")
print(susceptibility_bounds)

Flood bounds:
[[-1.3152685925803305, 36.789862088945036], [-1.259887695950126, 36.85027119777541]]

Susceptibility bounds:
[[-1.3152685925803305, 36.789862088945036], [-1.259887695950126, 36.85027119777541]]


In [42]:
# ============================================================
# URIP NAIROBI - SCENARIO-BASED URBAN INTELLIGENCE DASHBOARD
# ============================================================
#
# Scenarios:
#   1. Off-Peak
#   2. Peak Traffic
#   3. Flood + Traffic
#
# Dashboard workflow:
#   INCIDENT -> SCENARIO -> BEST EMERGENCY FACILITY ->
#   ALTERNATIVE ROUTES -> TRAVEL TIME + FLOOD EXPOSURE ->
#   OPERATIONAL RECOMMENDATION
#
# ------------------------------------------------------------
# WHAT CHANGED IN THIS VERSION (UI + base map only)
# ------------------------------------------------------------
# Sections 1-8 (imports + analytical engine: routing, alternative
# routes, flood exposure, recommendation logic) are UNCHANGED —
# only the map (base layers, markers, fit-to-scenario) and the
# dashboard UI (cards, table, legend, banner, toolbar) were
# redesigned, using the same navy/teal/coral system as the URIP
# presentation decks so the notebook, slides, and dashboard all
# read as one product.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import folium
from folium import plugins
from folium.plugins import BeautifyIcon
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from shapely.geometry import LineString
from rasterio.warp import transform_bounds


# ------------------------------------------------------------
# 2. PREPARE FLOOD GEOMETRY
# ------------------------------------------------------------

flood_union = flood_gdf.dissolve()
flood_geometry = flood_union.geometry.iloc[0]


# ------------------------------------------------------------
# 3. SAFE ROUTE -> GEODATAFRAME FUNCTION
# ------------------------------------------------------------

def route_to_gdf_safe(graph, route):
    """Convert a network route into a GeoDataFrame for visualization.
    Works with OSMnx MultiDiGraph networks."""

    route_edges = []

    for u, v in zip(route[:-1], route[1:]):
        try:
            edge_data = graph.get_edge_data(u, v)
            if edge_data is None:
                continue

            best_key = min(edge_data, key=lambda k: edge_data[k].get("length", float("inf")))
            data = edge_data[best_key]
            geometry = data.get("geometry")

            if geometry is None:
                node_u = graph.nodes[u]
                node_v = graph.nodes[v]
                geometry = LineString([(node_u["x"], node_u["y"]), (node_v["x"], node_v["y"])])

            route_edges.append({"u": u, "v": v, "geometry": geometry})
        except Exception:
            continue

    if not route_edges:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    return gpd.GeoDataFrame(route_edges, geometry="geometry", crs="EPSG:4326")


# ------------------------------------------------------------
# 4. CALCULATE ROUTE
# ------------------------------------------------------------

def calculate_scenario_route(graph, origin, destination, weight):
    try:
        route = nx.shortest_path(graph, origin, destination, weight=weight)
        travel_time = nx.shortest_path_length(graph, origin, destination, weight=weight)
        return route, travel_time
    except nx.NetworkXNoPath:
        return None, np.nan


# ------------------------------------------------------------
# 5. GENERATE ALTERNATIVE ROUTES
# ------------------------------------------------------------

def get_alternative_routes(graph, origin, destination, weight, k=3):
    """OSMnx uses a MultiDiGraph. NetworkX shortest_simple_paths does not
    support MultiDiGraph directly, so a temporary DiGraph is created
    containing the lowest-cost parallel edge."""

    simple_graph = nx.DiGraph()

    for u, v, key, data in graph.edges(keys=True, data=True):
        edge_weight = data.get(weight, float("inf"))
        if not np.isfinite(edge_weight):
            continue

        if simple_graph.has_edge(u, v):
            current_weight = simple_graph[u][v].get(weight, float("inf"))
            if edge_weight < current_weight:
                simple_graph[u][v].update(data)
        else:
            simple_graph.add_edge(u, v, **data)

    if origin not in simple_graph or destination not in simple_graph:
        return []

    alternatives = []
    try:
        paths = nx.shortest_simple_paths(simple_graph, origin, destination, weight=weight)
        for path in paths:
            time_sec = nx.path_weight(simple_graph, path, weight=weight)
            alternatives.append({"route": path, "time_sec": time_sec})
            if len(alternatives) >= k:
                break
    except nx.NetworkXNoPath:
        return []

    return alternatives


# ------------------------------------------------------------
# 6. CALCULATE FLOOD EXPOSURE OF A ROUTE
# ------------------------------------------------------------

def calculate_route_flood_exposure(graph, route, flood_geometry):
    """Flood exposure (%) = flooded route length / total route length x 100
    Classification: 0% = Not affected, 0-20% = Minor exposure,
    20-50% = Moderate disruption, >50% = Major disruption"""

    total_length = 0
    flooded_length = 0

    for u, v in zip(route[:-1], route[1:]):
        try:
            edge_data = graph.get_edge_data(u, v)
            if edge_data is None:
                continue

            best_edge = min(edge_data.values(), key=lambda x: x.get("length", float("inf")))
            geometry = best_edge.get("geometry")

            if geometry is None:
                node_u = graph.nodes[u]
                node_v = graph.nodes[v]
                geometry = LineString([(node_u["x"], node_u["y"]), (node_v["x"], node_v["y"])])

            edge_gdf = gpd.GeoSeries([geometry], crs="EPSG:4326").to_crs(flood_gdf.crs)
            projected_geometry = edge_gdf.iloc[0]

            flooded_segment = projected_geometry.intersection(flood_geometry)
            flooded_length += flooded_segment.length
            total_length += projected_geometry.length
        except Exception:
            continue

    if total_length == 0:
        return {"flooded_length_m": 0, "route_length_m": 0, "flood_percentage": 0, "flood_status": "Unknown"}

    flood_percentage = (flooded_length / total_length) * 100

    if flood_percentage == 0:
        status = "Not affected"
    elif flood_percentage <= 20:
        status = "Minor exposure"
    elif flood_percentage <= 50:
        status = "Moderate disruption"
    else:
        status = "Major disruption"

    return {
        "flooded_length_m": flooded_length,
        "route_length_m": total_length,
        "flood_percentage": flood_percentage,
        "flood_status": status,
    }


# ------------------------------------------------------------
# 7. ROUTE OPERATIONAL RECOMMENDATION
# ------------------------------------------------------------

def route_recommendation(routes_df, scenario):
    """Off-Peak / Peak Traffic: fastest route is recommended.
    Flood + Traffic: prefer the fastest route with <=20% flood exposure."""

    if routes_df.empty:
        return "NO ROUTE AVAILABLE"

    if scenario != "Flood + Traffic":
        return "USE FASTEST ROUTE"

    viable_routes = routes_df[routes_df["flood_percentage"] <= 20].copy()

    if not viable_routes.empty:
        best_viable = viable_routes.sort_values("time_sec").iloc[0]
        if best_viable["route_number"] == 1:
            return "USE FASTEST VIABLE ROUTE"
        else:
            return f"REROUTE VIA ROUTE {int(best_viable['route_number'])}"

    return "CAUTION: ALL ROUTES FLOOD-EXPOSED"


# ------------------------------------------------------------
# 8. SCENARIO ANALYSIS
# ------------------------------------------------------------

def analyse_scenario(incident_id, scenario, k_routes=3):
    """Main URIP analytical engine.
    Incident -> Scenario -> Best hospital -> Alternative routes -> Flood exposure"""

    incident_row = incidents_wgs84[incidents_wgs84["incident_id"] == incident_id]
    if incident_row.empty:
        return None

    incident_node = int(incident_row.iloc[0]["nearest_node"])

    if scenario == "Off-Peak":
        graph = G
        weight = "travel_time"
    elif scenario == "Peak Traffic":
        graph = G_traffic
        weight = "final_traffic_time"
    elif scenario == "Flood + Traffic":
        graph = G_combined
        weight = "combined_travel_time"
    else:
        return None

    hospital_results = []
    for _, hospital in hospitals_cbd.iterrows():
        hospital_name = hospital["name"]
        hospital_node = int(hospital["nearest_node"])
        try:
            travel_time = nx.shortest_path_length(graph, hospital_node, incident_node, weight=weight)
            hospital_results.append({"hospital": hospital_name, "hospital_node": hospital_node, "travel_time_sec": travel_time})
        except nx.NetworkXNoPath:
            continue

    if not hospital_results:
        return None

    hospital_df = pd.DataFrame(hospital_results).sort_values("travel_time_sec").reset_index(drop=True)
    best_hospital = hospital_df.iloc[0]
    best_hospital_name = best_hospital["hospital"]
    best_hospital_node = int(best_hospital["hospital_node"])

    alternatives = get_alternative_routes(graph, best_hospital_node, incident_node, weight, k=k_routes)

    route_results = []
    for i, alternative in enumerate(alternatives, start=1):
        route = alternative["route"]
        time_sec = alternative["time_sec"]

        flood_info = {"flooded_length_m": 0, "route_length_m": 0, "flood_percentage": 0, "flood_status": "Not assessed"}
        if scenario == "Flood + Traffic":
            flood_info = calculate_route_flood_exposure(graph, route, flood_geometry)

        route_results.append({
            "route_number": i, "route": route, "time_sec": time_sec, "time_min": time_sec / 60,
            "flooded_length_m": flood_info["flooded_length_m"], "route_length_m": flood_info["route_length_m"],
            "flood_percentage": flood_info["flood_percentage"], "flood_status": flood_info["flood_status"],
        })

    routes_df = pd.DataFrame(route_results)
    if routes_df.empty:
        return None

    best_route = routes_df.iloc[0]
    recommendation = route_recommendation(routes_df, scenario)

    return {
        "incident_id": incident_id, "incident_node": incident_node, "scenario": scenario,
        "hospital": best_hospital_name, "hospital_node": best_hospital_node,
        "hospital_time_min": best_hospital["travel_time_sec"] / 60,
        "hospital_rankings": hospital_df, "routes": routes_df,
        "best_route": best_route, "recommendation": recommendation,
    }


# ==============================================================
# 8.5 DESIGN SYSTEM
# --------------------------------------------------------------
# Same palette as the URIP presentation decks, so the notebook
# dashboard, the pitch deck, and the proposal deck all read as
# one visual product rather than three unrelated documents.
# ==============================================================

NAVY      = "#0B2545"
NAVY_2    = "#15335C"
TEAL      = "#13A9C7"
CORAL     = "#FF6B4A"
WHITE     = "#FFFFFF"
OFFWHITE  = "#F7F9FB"
TEXT_DARK = "#13294B"
MUTED     = "#5C6B7A"
CARD_BG   = "#EDF3F7"

# One shared classification -> color scale, used for BOTH the route
# table pills and the map line colors, so what you read in the table
# is the exact color you see on the map.
STATUS_COLORS = {
    "Not affected":        "#2A9D8F",
    "Minor exposure":      "#F4A261",
    "Moderate disruption": "#E63946",
    "Major disruption":    "#9D0208",
    "Unknown":             MUTED,
    "Not assessed":        MUTED,
}

def flood_status_from_pct(pct):
    """Same thresholds as calculate_route_flood_exposure, for
    road-segment-level (not route-level) coloring."""
    if pct <= 0:
        return "Not affected"
    elif pct <= 20:
        return "Minor exposure"
    elif pct <= 50:
        return "Moderate disruption"
    return "Major disruption"

def status_pill(status):
    color = STATUS_COLORS.get(status, MUTED)
    return (
        f'<span style="background:{color}22;color:{color};'
        f'border:1px solid {color}66;padding:3px 10px;border-radius:999px;'
        f'font-size:12px;font-weight:700;white-space:nowrap;">{status}</span>'
    )

def summary_card(icon, label, value, accent=TEAL):
    return f"""
    <div style="flex:1;background:{WHITE};border-radius:10px;padding:16px 18px;
                border-left:4px solid {accent};
                box-shadow:0 1px 4px rgba(11,37,69,0.10);">
        <div style="font-size:11.5px;font-weight:700;letter-spacing:0.5px;
                    color:{MUTED};text-transform:uppercase;margin-bottom:6px;">
            {icon}&nbsp; {label}
        </div>
        <div style="font-size:19px;font-weight:700;color:{TEXT_DARK};line-height:1.3;">
            {value}
        </div>
    </div>
    """

# (prefix, (icon, color)) — checked in order, first match wins
_RECOMMENDATION_STYLE = [
    ("USE FASTEST VIABLE ROUTE", ("\u2705", "#2A9D8F")),
    ("USE FASTEST ROUTE",        ("\u2705", "#2A9D8F")),
    ("REROUTE",                  ("\u21aa",  "#F4A261")),
    ("CAUTION",                  ("\u26a0",  "#E63946")),
    ("NO ROUTE AVAILABLE",       ("\u26d4",  "#9D0208")),
]

def recommendation_style(text):
    for prefix, style in _RECOMMENDATION_STYLE:
        if text.startswith(prefix):
            return style
    return ("\u2139", TEAL)


# ------------------------------------------------------------
# 9. FLOOD RASTER BOUNDS FOR FOLIUM
# ------------------------------------------------------------

flood_bounds_wgs84 = transform_bounds(flood_raster.crs, "EPSG:4326", *flood_raster.bounds)
susceptibility_bounds_wgs84 = transform_bounds(susceptibility_raster.crs, "EPSG:4326", *susceptibility_raster.bounds)


# ------------------------------------------------------------
# 10. CREATE FLOOD SUSCEPTIBILITY OVERLAY
# ------------------------------------------------------------

sus_data = susceptibility_raster.read(1)
sus_masked = np.ma.masked_invalid(sus_data)
sus_norm = np.clip(sus_masked, 0, 1)
sus_rgba = plt.cm.YlOrRd(sus_norm.filled(0))
sus_rgba[..., 3] = np.where(sus_masked.mask, 0, 0.60)


# ------------------------------------------------------------
# 11. CREATE FLOOD SCENARIO OVERLAY
# ------------------------------------------------------------

flood_data = flood_raster.read(1)
flood_mask = (flood_data == 1)
flood_rgba = np.zeros((flood_data.shape[0], flood_data.shape[1], 4))
flood_rgba[flood_mask] = [0.90, 0.13, 0.27, 0.55]  # brand danger red (#E63946), not pure CSS red


# ------------------------------------------------------------
# 12. PREPARE AFFECTED ROADS
# ------------------------------------------------------------

affected_roads = edges_projected[edges_projected["flood_percentage"] > 0].copy()
major_flood_roads = edges_projected[edges_projected["flood_percentage"] > 50].copy()


# ==============================================================
# 12.5 BASE MAP BUILDER  (this is the "base map" improvement)
# --------------------------------------------------------------
# - Three switchable base layers (Light / Dark / Satellite)
#   instead of one fixed OSM tile
# - Fullscreen + MiniMap for actual field/ops usability
# - Auto-fits to the incident + facility + all routes, instead
#   of a fixed zoom centered only on the incident (which could
#   previously crop the hospital or routes out of view)
# - Brand-colored BeautifyIcon markers instead of default
#   Leaflet red/green pins
# ==============================================================

def build_base_map(center, bounds=None):
    m = folium.Map(location=center, zoom_start=14, control_scale=True, tiles=None)

    # NOTE: CartoDB's hosted "positron"/"dark_matter" tiles now require an API
    # key (as of CARTO's basemap policy change), so we use Esri's free,
    # no-key-required tile services instead. The Light/Dark "gray canvas"
    # styles are intentionally minimal — they're designed as a plain backdrop
    # so the flood zone, routes, and infrastructure markers stay the visual
    # focus rather than competing with dense street labels. "Streets" and
    # "Satellite" are there for when the responder needs full place-name
    # context instead.
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri", name="Light", control=True,
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri", name="Dark", control=True,
    ).add_to(m)
    folium.TileLayer("OpenStreetMap", name="Streets", control=True).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri", name="Satellite", control=True,
    ).add_to(m)

    plugins.Fullscreen(position="topleft", title="Fullscreen", title_cancel="Exit fullscreen").add_to(m)
    plugins.MiniMap(toggle_display=True, position="bottomright", zoom_level_offset=-5).add_to(m)

    if bounds:
        m.fit_bounds(bounds, padding=(30, 30))

    return m


def beautified_marker(lat, lon, icon, color, popup=None, tooltip=None):
    return folium.Marker(
        [lat, lon], popup=popup, tooltip=tooltip,
        icon=BeautifyIcon(icon=icon, icon_shape="marker", border_color=color,
                           text_color=WHITE, background_color=color),
    )


def compute_scenario_bounds(incident_latlon, hospital_latlon, route_gdfs):
    """Lat/lon bounding box covering the incident, the facility, and every
    candidate route, so the map always frames the full scenario."""
    lats = [incident_latlon[0], hospital_latlon[0]]
    lons = [incident_latlon[1], hospital_latlon[1]]
    for gdf in route_gdfs:
        if gdf.empty:
            continue
        b = gdf.total_bounds  # minx, miny, maxx, maxy
        lons += [b[0], b[2]]
        lats += [b[1], b[3]]
    return [[min(lats), min(lons)], [max(lats), max(lons)]]


# ------------------------------------------------------------
# 13. DASHBOARD WIDGETS
# ------------------------------------------------------------

incident_input = widgets.IntText(
    value=35, description="\U0001F4CD Incident:", min=1, max=50,
    style={"description_width": "initial"}, layout=widgets.Layout(width="180px"),
)

scenario_dropdown = widgets.Dropdown(
    options=["Off-Peak", "Peak Traffic", "Flood + Traffic"],
    value="Flood + Traffic", description="\U0001F30A Scenario:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="260px"),
)

analyse_button = widgets.Button(
    description="Analyse Incident", button_style="primary", icon="search",
    layout=widgets.Layout(width="180px"),
)

status_label = widgets.HTML(value="")
dashboard_output = widgets.Output()


# ------------------------------------------------------------
# 14. DISPLAY DASHBOARD
# ------------------------------------------------------------

def display_dashboard(incident_id, scenario):

    with dashboard_output:
        clear_output(wait=True)

        # ------------------------------------------------
        # Validate incident
        # ------------------------------------------------
        if incident_id < 1 or incident_id > 50:
            display(HTML(f"""
                <div style="padding:14px 16px;background:#FDEDEE;color:#9D0208;
                            border-left:4px solid #E63946;border-radius:6px;">
                    <b>Invalid incident number.</b> Please select an incident between 1 and 50.
                </div>
            """))
            return

        analysis = analyse_scenario(incident_id, scenario, k_routes=3)

        if analysis is None:
            display(HTML(f"""
                <div style="padding:14px 16px;background:#FDEDEE;color:#9D0208;
                            border-left:4px solid #E63946;border-radius:6px;">
                    <b>No viable route or facility found for this scenario.</b>
                </div>
            """))
            return

        # ------------------------------------------------
        # Extract results
        # ------------------------------------------------
        incident_row = incidents_wgs84[incidents_wgs84["incident_id"] == incident_id].iloc[0]
        incident_lat, incident_lon = incident_row.geometry.y, incident_row.geometry.x

        hospital_name = analysis["hospital"]
        hospital_row = hospitals_cbd[hospitals_cbd["name"] == hospital_name].iloc[0]
        hospital_lat, hospital_lon = hospital_row.geometry.y, hospital_row.geometry.x

        routes_df = analysis["routes"].copy()
        recommendation = analysis["recommendation"]
        best_time = analysis["hospital_time_min"]

        # ------------------------------------------------
        # DASHBOARD HEADER  (navy, matches deck title slides)
        # ------------------------------------------------
        display(HTML(f"""
            <div style="padding:18px 22px;border-radius:10px;background:{NAVY};
                        margin-bottom:14px;">
                <div style="color:{TEAL};font-size:11px;font-weight:700;
                            letter-spacing:1.5px;margin-bottom:4px;">
                    URIP &middot; URBAN EMERGENCY INTELLIGENCE
                </div>
                <h2 style="margin:0;color:{WHITE};font-size:22px;">
                    Incident {incident_id}
                    <span style="color:#8FA3B8;font-weight:400;"> &middot; {scenario}</span>
                </h2>
            </div>
        """))

        # ------------------------------------------------
        # SUMMARY CARDS
        # ------------------------------------------------
        display(HTML(f"""
            <div style="display:flex;gap:12px;margin-bottom:16px;">
                {summary_card("\U0001F3E5", "Best Facility", hospital_name, accent=TEAL)}
                {summary_card("\u23F1", "Fastest Response", f"{best_time:.2f} min", accent=NAVY)}
                {summary_card("\U0001F4E1", "Operational Action", recommendation,
                               accent=recommendation_style(recommendation)[1])}
            </div>
        """))

        # ------------------------------------------------
        # BUILD MAP
        # ------------------------------------------------
        graph_for_scenario = G if scenario == "Off-Peak" else G_traffic if scenario == "Peak Traffic" else G_combined

        # Pre-build every route's geometry once, so we can both draw them
        # AND fit the map bounds to cover all of them.
        route_gdfs = {}
        for idx, route_row in routes_df.iterrows():
            route_gdfs[int(route_row["route_number"])] = route_to_gdf_safe(graph_for_scenario, route_row["route"])

        scenario_bounds = compute_scenario_bounds(
            (incident_lat, incident_lon), (hospital_lat, hospital_lon), list(route_gdfs.values())
        )

        m = build_base_map(center=[incident_lat, incident_lon], bounds=scenario_bounds)

        # ------------------------------------------------
        # FLOOD LAYERS  (Flood + Traffic only)
        # ------------------------------------------------
        if scenario == "Flood + Traffic":

            folium.raster_layers.ImageOverlay(
                image=sus_rgba,
                bounds=[[flood_bounds_wgs84[1], flood_bounds_wgs84[0]], [flood_bounds_wgs84[3], flood_bounds_wgs84[2]]],
                opacity=0.60, interactive=True, cross_origin=False, zindex=1,
                name="Flood Susceptibility",
            ).add_to(m)

            folium.raster_layers.ImageOverlay(
                image=flood_rgba,
                bounds=[[flood_bounds_wgs84[1], flood_bounds_wgs84[0]], [flood_bounds_wgs84[3], flood_bounds_wgs84[2]]],
                opacity=0.55, interactive=True, cross_origin=False, zindex=2,
                name="Flood Scenario",
            ).add_to(m)

            affected_layer = folium.FeatureGroup(name="Flood-Affected Roads")
            for _, row in affected_roads.iterrows():
                if row.geometry is None:
                    continue
                status = flood_status_from_pct(row["flood_percentage"])
                line_color = STATUS_COLORS[status]
                folium.GeoJson(
                    row.geometry.__geo_interface__,
                    style_function=lambda feature, c=line_color: {"color": c, "weight": 2, "opacity": 0.7},
                    tooltip=f"Flood exposure: {row['flood_percentage']:.1f}% ({status})",
                ).add_to(affected_layer)
            affected_layer.add_to(m)

        # ------------------------------------------------
        # EMERGENCY INFRASTRUCTURE MARKERS
        # ------------------------------------------------
        beautified_marker(
            incident_lat, incident_lon, icon="exclamation-triangle", color=CORAL,
            popup=f"<b>Incident {incident_id}</b><br>Simulated Emergency",
            tooltip=f"Incident {incident_id}",
        ).add_to(m)

        beautified_marker(
            hospital_lat, hospital_lon, icon="plus", color=NAVY,
            popup=(f"<b>{hospital_name}</b><br>Best facility for {scenario}"
                   f"<br>Response time: {best_time:.2f} min"),
            tooltip=f"Best facility: {hospital_name}",
        ).add_to(m)

        # ------------------------------------------------
        # ROUTE LAYERS
        # ------------------------------------------------
        # Off-peak/peak routes get distinct brand colors; flood-scenario
        # routes are colored by flood exposure (same scale as the table).
        rank_colors = [TEAL, CORAL, NAVY_2]

        for idx, route_row in routes_df.iterrows():
            route_number = int(route_row["route_number"])
            route_time = route_row["time_min"]
            flood_pct = route_row["flood_percentage"]
            flood_status = route_row["flood_status"]

            if scenario == "Flood + Traffic":
                route_color = STATUS_COLORS.get(flood_status, MUTED)
            else:
                route_color = rank_colors[min(route_number - 1, len(rank_colors) - 1)]

            route_gdf = route_gdfs[route_number]
            route_layer = folium.FeatureGroup(name=f"Route {route_number} \u2014 {route_time:.2f} min")

            for _, edge in route_gdf.iterrows():
                folium.GeoJson(
                    edge.geometry.__geo_interface__,
                    style_function=lambda feature, c=route_color: {"color": c, "weight": 6, "opacity": 0.9},
                    tooltip=(f"Route {route_number}<br>Travel time: {route_time:.2f} min"
                             f"<br>Flood exposure: {flood_pct:.1f}%<br>Flood status: {flood_status}"),
                ).add_to(route_layer)

            route_layer.add_to(m)

        # ------------------------------------------------
        # LEGEND  (only meaningful for the flood scenario)
        # ------------------------------------------------
        if scenario == "Flood + Traffic":
            legend_rows = "".join(
                f'<div style="margin:3px 0;"><span style="display:inline-block;width:14px;height:4px;'
                f'background:{color};margin-right:8px;border-radius:2px;"></span>{status}</div>'
                for status, color in STATUS_COLORS.items()
                if status not in ("Unknown", "Not assessed")
            )
            legend_html = f"""
            <div style="position:fixed;bottom:30px;left:30px;z-index:9999;
                        background:{WHITE};padding:12px 14px;border-radius:8px;
                        box-shadow:0 2px 8px rgba(11,37,69,0.25);font-size:12px;
                        color:{TEXT_DARK};font-family:sans-serif;">
                <div style="font-weight:700;margin-bottom:6px;">Route / Road Flood Exposure</div>
                {legend_rows}
                <hr style="margin:8px 0;border:none;border-top:1px solid {CARD_BG};">
                <div><span style="display:inline-block;width:12px;height:12px;
                     background:{STATUS_COLORS['Moderate disruption']};opacity:0.6;
                     margin-right:8px;border-radius:2px;"></span>Flood scenario extent</div>
            </div>
            """
            m.get_root().html.add_child(folium.Element(legend_html))

        folium.LayerControl(collapsed=False).add_to(m)
        display(m)

        # ------------------------------------------------
        # ROUTE COMPARISON TABLE  (styled HTML, not a raw dataframe)
        # ------------------------------------------------
        display(HTML(f'<h3 style="color:{TEXT_DARK};margin-bottom:8px;">Route Comparison</h3>'))

        rows_html = ""
        for _, r in routes_df.iterrows():
            is_flood = scenario == "Flood + Traffic"
            status_cell = status_pill(r["flood_status"]) if is_flood else '<span style="color:#AAB4BE;">N/A</span>'
            flood_pct_cell = f"{r['flood_percentage']:.1f}%" if is_flood else "—"
            flooded_len_cell = f"{r['flooded_length_m']:.0f} m" if is_flood else "—"
            highlight = f"background:{CARD_BG};" if int(r["route_number"]) == 1 else ""
            rows_html += f"""
            <tr style="{highlight}">
                <td style="padding:9px 12px;font-weight:700;">Route {int(r['route_number'])}</td>
                <td style="padding:9px 12px;">{r['time_min']:.2f} min</td>
                <td style="padding:9px 12px;">{r['route_length_m']:.0f} m</td>
                <td style="padding:9px 12px;">{flooded_len_cell}</td>
                <td style="padding:9px 12px;">{flood_pct_cell}</td>
                <td style="padding:9px 12px;">{status_cell}</td>
            </tr>
            """

        display(HTML(f"""
        <table style="width:100%;border-collapse:collapse;font-size:13.5px;
                      font-family:sans-serif;color:{TEXT_DARK};
                      box-shadow:0 1px 4px rgba(11,37,69,0.08);border-radius:8px;overflow:hidden;">
            <thead>
                <tr style="background:{NAVY};color:{WHITE};text-align:left;">
                    <th style="padding:10px 12px;">Route</th>
                    <th style="padding:10px 12px;">Travel Time</th>
                    <th style="padding:10px 12px;">Route Length</th>
                    <th style="padding:10px 12px;">Flooded Length</th>
                    <th style="padding:10px 12px;">Flood Exposure</th>
                    <th style="padding:10px 12px;">Flood Status</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
        """))

        # ------------------------------------------------
        # OPERATIONAL RECOMMENDATION BANNER  (dynamic color)
        # ------------------------------------------------
        rec_icon, rec_color = recommendation_style(recommendation)
        display(HTML(f"""
            <div style="padding:16px 18px;margin-top:14px;border-radius:8px;
                        background:{rec_color}14;border-left:4px solid {rec_color};">
                <h3 style="margin:0 0 6px 0;color:{TEXT_DARK};">
                    {rec_icon} Operational Recommendation
                </h3>
                <p style="margin:0 0 6px 0;font-size:16px;font-weight:700;color:{rec_color};">
                    {recommendation}
                </p>
                <p style="margin:0;color:{MUTED};font-size:13px;">
                    The system evaluates travel time together with network conditions and,
                    under the flood scenario, route-level flood exposure.
                </p>
            </div>
        """))


# ------------------------------------------------------------
# 15. BUTTON CALLBACK
# ------------------------------------------------------------

def analyse_button_clicked(button):
    incident_id = incident_input.value
    scenario = scenario_dropdown.value

    status_label.value = f'<span style="color:{MUTED};">\u23F3 Analysing...</span>'

    try:
        display_dashboard(incident_id, scenario)
        status_label.value = f'<span style="color:#2A9D8F;font-weight:600;">\u2713 Analysis complete.</span>'
    except Exception as e:
        status_label.value = f'<span style="color:#E63946;font-weight:600;">Error: {str(e)}</span>'
        with dashboard_output:
            print("Dashboard error:")
            print(e)


analyse_button.on_click(analyse_button_clicked)


# ------------------------------------------------------------
# 16. DASHBOARD UI
# ------------------------------------------------------------

dashboard_title = widgets.HTML(value=f"""
    <div style="padding:14px 0 10px 0;">
        <div style="color:{TEAL};font-size:11px;font-weight:700;letter-spacing:1.5px;">
            URIP &middot; NAIROBI
        </div>
        <h2 style="margin:2px 0 0 0;color:{TEXT_DARK};">Emergency Response Dashboard</h2>
        <p style="margin:4px 0 0 0;color:{MUTED};font-size:13.5px;">
            Select an incident and a scenario to evaluate the best emergency response route.
        </p>
    </div>
""")

controls = widgets.HBox(
    [incident_input, scenario_dropdown, analyse_button],
    layout=widgets.Layout(
        align_items="center", justify_content="flex-start",
        border_bottom="1px solid #DCE4EA", padding="4px 0 16px 0", margin="0 0 10px 0",
    ),
)

dashboard_ui = widgets.VBox([dashboard_title, controls, status_label, dashboard_output])

display(dashboard_ui)


# ------------------------------------------------------------
# 17. INITIAL DEMONSTRATION
# ------------------------------------------------------------

display_dashboard(incident_id=35, scenario="Flood + Traffic")